In [288]:
# Define geometry and mesh for the FEM

from netgen.occ import *
from netgen.meshing import IdentificationType
from ngsolve import *
from ngsolve.webgui import Draw

# 1. Geometry Setup
L = 4.0
Lx = L/8
Ly = L/128
Lz_1 = 0.69 * L       # Height of bottom air volume
Lz_plate = 0.01 * L  # Thickness of the elastic plate
Lz_2 = 0.3 * L      # Height of top air volume
Lz = Lz_1 + Lz_plate + Lz_2

# 1. Bottom Air Volume (from z=0 to z=Lz_1)
air_bottom = Box((0, 0, 0), (Lx, Ly, Lz_1))
air_bottom.mat("air")
air_bottom.faces.Min(Z).name = "bottom"

# 2. Elastic Plate (from z=Lz_1 to z=Lz_1+Lz_plate)
z_plate_top = Lz_1 + Lz_plate
plate = Box((0, 0, Lz_1), (Lx, Ly, z_plate_top))
plate.mat("plate")
plate.faces.Min(Z).name = "fsi_bottom"
plate.faces.Max(Z).name = "fsi_top"

# 3. Top Air Volume (from z=plate_top to z=Lz)
air_top = Box((0, 0, z_plate_top), (Lx, Ly, Lz))
air_top.mat("air")
air_top.faces.Max(Z).name = "top"

Lz_viz = 0.5 * L
wbm_viz_box = Box((0, 0, Lz), (Lx, Ly, Lz + Lz_viz))
wbm_viz_box.mat("wbm_viz") # Give it a unique material name

# Glue them together to ensure conforming mesh nodes at the interfaces
geometry = Glue([air_bottom, plate, air_top, wbm_viz_box])

# 3. Identify Periodic Faces (on the outer boundaries of the glued geometry)
geometry.faces.Max(X).Identify(geometry.faces.Min(X), "periodic_x")
geometry.faces.Max(Y).Identify(geometry.faces.Min(Y), "periodic_y")

# 4. Generate Mesh
geo = OCCGeometry(geometry)
mesh = Mesh(geo.GenerateMesh(maxh=0.1))

print("Mesh generated successfully! Notice the internal face at z=0.2")
Draw(mesh)

Mesh generated successfully! Notice the internal face at z=0.2


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [289]:
# Define physics
import cmath, math
# 1. Physics Parameters
freq = 343.0     
c_0 = 343.0* (1 - 1j * 0.001)  # Complex speed of sound to introduce a small amount of damping
k = 2 * math.pi * freq / c_0  
rho_air = 1.21
omega = 2 * math.pi * freq

# Solid (Steel Plate) Parameters
E = 210e9              # Young's Modulus (Pa)
nu = 0.3               # Poisson's ratio
rho_s = 7800.0/1e4         # Density (kg/m^3)
mu = E / (2 * (1 + nu))
lam = E * nu / ((1 + nu) * (1 - 2 * nu))

# Incident angles (e.g., 45 degrees)
theta = (math.pi / 4)  # Polar angle (0 for normal incidence)
phi = 0.0

# Wave vector components
kx = k * math.sin(theta) * math.cos(phi)
ky = k * math.sin(theta) * math.sin(phi)
kz = k * math.cos(theta)

# Transverse wave vector for the Floquet shift
k_vec = CF((kx, ky, 0)) 

# Finite Element Space
fes_air = Periodic(H1(mesh, order=3, complex=True, definedon=mesh.Materials("air")))
fes_plate = Periodic(VectorH1(mesh, order=3, complex=True, definedon=mesh.Materials("plate")))

fes = fes_air * fes_plate
(p, u), (q, w) = fes.TnT()  # p,q = acoustic ; u,w = elastic

# --- Strain/Stress definitions for Elasticity ---
def eps(vec): return Sym(Grad(vec))
def sigma(vec): return 2*mu*eps(vec) + lam*Trace(eps(vec))*Id(3)

print(f"Total FSI Degrees of Freedom: {fes.ndof}")
print(f"wave vector k: ({kx:.2f}, {ky:.2f}, {kz:.2f})")


Total FSI Degrees of Freedom: 13881
wave vector k: (4.44+0.00j, 0.00+0.00j, 4.44+0.00j)


In [290]:
# FEM model: shift cell operator method (Floquet trick)
# --- Modified Floquet Operators ---
def grad_p(p): return grad(p) + 1j * k_vec * p
def grad_q(q): return grad(q) - 1j * k_vec * q
def grad_u(u): return grad(u) + 1j * OuterProduct(u, k_vec)
def grad_w(w): return grad(w) - 1j * OuterProduct(w, k_vec)
def eps_u(u): return Sym(grad_u(u))
def eps_w(w): return Sym(grad_w(w))
def sigma(vec): return 2*mu*eps_u(vec) + lam*Trace(eps_u(vec))*Id(3)

# 2. Bilinear Form (LHS)
Z_fem = BilinearForm(fes)
Z_fem += (InnerProduct(grad_p(p), grad_q(q)) - k**2 * p * q) * dx("air")
Z_fem += (InnerProduct(sigma(u), eps_w(w)) - rho_s * omega**2 * InnerProduct(u, w)) * dx("plate")

# FSI Interface Coupling 
Z_fem += ( rho_air * omega**2 * u[2] * q) * ds("fsi_top")    
Z_fem += ( p * w[2]) * ds("fsi_top")                         
Z_fem += (-rho_air * omega**2 * u[2] * q) * ds("fsi_bottom") 
Z_fem += (-p * w[2]) * ds("fsi_bottom")

# 3. Linear Form (RHS) - Surface source on the bottom boundary
s_fem = LinearForm(fes)
source_func = exp(-(z - Lz_1*0.8)**2 / (2 * (Lz_1/10)**2))  # Gaussian source centered at z=Lz_1/2
s_fem += source_func* q * dx("air")

# 4. Assemble 
with TaskManager():
    Z_fem.Assemble()
    s_fem.Assemble()
    
# 5. Extract free DOFs
freedofs = fes.FreeDofs()
free_indices = [i for i, is_free in enumerate(freedofs) if is_free]
slave_indices = [i for i, is_free in enumerate(freedofs) if not is_free]
print("Assembly complete!")


Assembly complete!


In [291]:
# Define wave functions for the Wave Based Method (WBM)
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla

nWave_x = 10 
nWave_y = 0 

m_indices = np.arange(-nWave_x, nWave_x + 1) # Harmonics in x: -nWave_x, ..., nWave_x
n_indices = np.arange(-nWave_y, nWave_y + 1) # Harmonics in y: -nWave_y, ..., nWave_y
total_waves = len(m_indices) * len(n_indices)

wave_kx = []
wave_ky = []
wave_kz = []
wave_functions = []

for m in m_indices:
    for n in n_indices:
        # Transverse Bloch-Floquet wavenumbers
        kx_wbm = 2 * np.pi * m / Lx
        ky_wbm = 2 * np.pi * n / Ly
        
        # Calculate axial wavenumber (kz), ensuring proper decay for evanescent waves
        val = k**2 - (kx + kx_wbm)**2 - (ky + ky_wbm)**2
        if val.real >= 0:
            kz_wbm = np.sqrt(val)
        else:
            kz_wbm = 1j * np.sqrt(-val) # Decay in +z direction
            
        wave_kx.append(kx_wbm)
        wave_ky.append(ky_wbm)
        wave_kz.append(kz_wbm)
        
        # Define Phi_w symbolically in NGSolve
        truncate_fac = IfPos(z-Lz-0.1, 1, 0) # Only visualize WBM contribution above the original domain
        phi_w = exp(1j * (kx_wbm * x + ky_wbm * y + kz_wbm * truncate_fac* (z - Lz)))
        wave_functions.append(phi_w)

print(f"Total waves in WBM: {total_waves}")

Total waves in WBM: 21


In [292]:
# Assemble WBM Matrices
print(f"Assembling WBM Coupling Matrices for {total_waves} Bloch-Floquet waves...")
Z_wbm = np.zeros((total_waves, total_waves), dtype=complex)
Z_hyb = np.zeros((fes.ndof, total_waves), dtype=complex)
n_wb = CF((0, 0, -1))
for i in range(total_waves):
    dphi_i_dz = 1j*wave_kz[i]*wave_functions[i]
    cwf = LinearForm(fes)
    cwf += - dphi_i_dz * q * ds("top")
    with TaskManager():
        cwf.Assemble()
    Z_hyb[:, i] = cwf.vec.FV().NumPy()
    for j in range(total_waves):
        phi_j = wave_functions[j]
        integrand = Conj(dphi_i_dz) * phi_j
        
        # Integrate symbolically over the interface boundary
        val = Integrate(integrand, mesh, definedon=mesh.Boundaries("top"))
        Z_wbm[i, j] = val

print("shape of Z_wbm:", Z_wbm.shape)
print("shape of Z_hyb:", Z_hyb.shape)
print("condition number of Z_wbm:", np.linalg.cond(Z_wbm))

Assembling WBM Coupling Matrices for 21 Bloch-Floquet waves...
shape of Z_wbm: (21, 21)
shape of Z_hyb: (13881, 21)
condition number of Z_wbm: 31.091788635861494


In [293]:
# Hybrid model

# Convert sparse Z_fem to SciPy CSC format
row, col, val = Z_fem.mat.COO()
Z_fem_scipy = sp.coo_matrix((val, (row, col)), shape=(fes.ndof, fes.ndof)).tocsc()
Z_fem_free = Z_fem_scipy[free_indices, :][:, free_indices]
Z_hyb_free = Z_hyb[free_indices, :]

f_f_np = s_fem.vec.FV().NumPy()
s_fem_free = f_f_np[free_indices]

# Block Matrix 
top_row = sp.hstack([Z_fem_free, Z_hyb_free])
bottom_row = np.hstack([Z_hyb_free.conj().T, Z_wbm])
Global_Matrix = sp.vstack([top_row, bottom_row]).tocsc()

# Global RHS Vector 
s_wbm = np.zeros(total_waves, dtype=complex)
Global_RHS = np.concatenate([s_fem_free, s_wbm])

print(f"Is Global_Matrix sparse? {sp.issparse(Global_Matrix)}")
print(f"Exact type of Global_Matrix: {type(Global_Matrix)}")
# print("condition number of Global_Matrix:", np.linalg.cond(Global_Matrix.toarray()))

Is Global_Matrix sparse? True
Exact type of Global_Matrix: <class 'scipy.sparse._csc.csc_matrix'>


In [294]:
# Solve the dense/sparse hybrid system using SuperLU
solution = spla.spsolve(Global_Matrix, Global_RHS)

p_fem_vals = solution[:len(free_indices)]
p_wbm_factors = solution[len(free_indices):]

# Map back to an NGSolve GridFunction
gf_env = GridFunction(fes)
gf_env.vec.FV().NumPy()[free_indices] = p_fem_vals
P_env_gf, U_env_gf = gf_env.components

In [295]:
# 1. Reconstruct Field
phase = exp(1j * (kx * x + ky * y))
p_true = P_env_gf * phase
u_true = U_env_gf * phase

p_wbm_total = sum([CF(complex(p_wbm_factors[i])) * phase* wave_functions[i] for i in range(total_waves)])

truncate_fac = IfPos(z-Lz, 1, 0)

# 2. Visualize
print("Rendering total pressure field ...")
Draw(truncate_fac* p_wbm_total + (1-truncate_fac)*p_true, mesh, "Acoustic Pressure", animate_complex=True)
Draw(u_true[2], mesh, "Plate Displacement", deformation=True, animate=True, scale=2e7)

Rendering total pressure field ...


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 's…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 's…

BaseWebGuiScene